# GCN / GAT / MLP on Fixed Pre-Generated Splits

Uses the **same 200 pre-generated splits** as other experiments (HetMap etc.) for direct comparison.
Results flushed every `FLUSH_EVERY` runs. Per-file misclassification rates saved alongside main results.

**Edit Cell 1, then Run All.**

In [1]:
# ── Cell 1: CONFIG — loaded from experiments/config.json ─────────────────────
import json
from pathlib import Path

_cfg = json.loads((Path.cwd() / "experiments" / "config.json").read_text())

SPLITS_DIR = r"C:\Users\JABEERAK\Architecture_Recovery\HetMap\data\splits"
DATA_DIR   = r"C:\Users\JABEERAK\Architecture_Recovery\data\datasets"

# Which models to run — edit in config.json → "models_to_run"
MODELS = _cfg["models_to_run"]

# All ablations are run for GCN/GAT (used in the paper): hetero_directed,
# homo_directed, hetero_undirected, hetero_reversed, hetero_directed_1layer.
# Each gets its own results CSV.
ABLATIONS = _cfg["ablations"]
ABLATION_SUFFIX = {
    "hetero_directed":        "hetero",
    "homo_directed":          "homo",
    "hetero_undirected":      "undirected",
    "hetero_reversed":        "reversed",
    "hetero_directed_1layer": "1L",
}

# Per-model architecture params (GCN has no heads, MLP has no mode)
MODEL_PARAMS = _cfg["model_params"]

# Training params
_t                = _cfg["training"]
LR                = _t["lr"]
THRESHOLD         = _t["threshold"]
WARMUP_EPOCHS     = _t["warmup_epochs"]
SELF_TRAIN_ROUNDS = _t["self_train_rounds"]
SELF_TRAIN_EPOCHS = _t["self_train_epochs"]

# Run control
RUN_IDS   = list(range(100))   # change range here for quick tests
DATASETS  = "all"              # "all" or e.g. ["ant", "jabref"]
FLUSH_EVERY = _cfg.get("flush_every", 5)

RESULTS_DIR   = _cfg.get("results_dir", "results")
NODE_PRED_DIR = f"{RESULTS_DIR}/node_predictions"


def results_file_for(model_type, ablation_name=None):
    """Per-model/per-ablation results CSV. MLP is edge-agnostic and shares one file."""
    if model_type == "MLP":
        return f"{RESULTS_DIR}/mlp_results.csv"
    return f"{RESULTS_DIR}/{model_type.lower()}_{ABLATION_SUFFIX[ablation_name]}_results.csv"


print(f"Data dir : {DATA_DIR}")
print(f"Ablations: {[a['name'] for a in ABLATIONS]}")
print(f"Models   : {MODELS}")
print(f"Training : lr={LR}  threshold={THRESHOLD}  warmup={WARMUP_EPOCHS}"
      f"  rounds={SELF_TRAIN_ROUNDS}×{SELF_TRAIN_EPOCHS}ep")
print(f"Model params: {MODEL_PARAMS}")

Data dir : C:\Users\JABEERAK\Architecture_Recovery\data\datasets
Ablations: ['hetero_directed', 'homo_directed', 'hetero_undirected', 'hetero_directed_1layer']
Models   : ['GCN', 'GAT', 'MLP']
Training : lr=0.001  threshold=0.95  warmup=100  rounds=4×30ep
Model params: {'GCN': {'hidden': 256, 'num_layers': 2, 'dropout': 0.01}, 'GAT': {'hidden': 256, 'num_layers': 2, 'heads': 4, 'dropout': 0.01}, 'MLP': {'hidden': 256, 'dropout': 0.0}}


In [2]:
# ── Cell 2: IMPORTS ───────────────────────────────────────────────────────────
import sys, json, copy
from pathlib import Path

import numpy as np
import torch
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from torch_geometric.data import HeteroData

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data_pipeline").is_dir() and (p / "experiments").is_dir():
            return p
    raise RuntimeError(
        f"Could not locate 'data_pipeline/' + 'experiments/' above {start}. "
        "Restart the kernel and Run All from the top of this notebook."
    )

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_pipeline import datasets, features_w2v, loc_features, graph
from models.gcn import RelationalGCN
from models.gat import RelationalGAT
from models.mlp import MLP
from training.self_train import self_train

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
Path(NODE_PRED_DIR).mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

Project root: c:\Users\JABEERAK\Architecture_Recovery\c2a_ext\c2a_mapping\c2a_mapping
Device: cuda


In [3]:
# ── Cell 3: UTILITIES ─────────────────────────────────────────────────────────

DISPLAY_TO_STEM = {
    "Ant": "ant", "A.UML": "argouml", "C.Img": "commons",
    "Jabref": "jabref", "Lucene": "lucene", "SH-3D": "sweetHome",
    "TeamMates": "teammates", "Bash": "bash",
    "HDF": "hdf", "HDC": "hdc", "Chrome": "chromium",
}
STEM_TO_DISPLAY = {v: k for k, v in DISPLAY_TO_STEM.items()}


def load_splits(stem):
    display = STEM_TO_DISPLAY[stem]
    path = Path(SPLITS_DIR) / f"{display}_splits.json"
    if not path.exists():
        raise FileNotFoundError(f"No splits file for {stem} at {path}")
    with open(path) as f:
        return json.load(f)


def split_to_index(train_files, file_df):
    """Convert train file path list -> LongTensor of node indices."""
    file_to_idx = {str(f): i for i, f in enumerate(file_df["File"])}
    return torch.tensor(
        [file_to_idx[f] for f in train_files if f in file_to_idx], dtype=torch.long
    )


def flush(rows, out_path):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    p = Path(out_path)
    if p.exists():
        df_new = pd.concat([pd.read_csv(p), df_new], ignore_index=True).drop_duplicates(
            subset=["data_name", "model", "mode", "directed", "reverse_relations", "run_id"],
            keep="first",
        )
    df_new.to_csv(p, index=False)


def load_misclas(out_path, n_nodes, files):
    wrong  = np.zeros(n_nodes, dtype=np.int32)
    tested = np.zeros(n_nodes, dtype=np.int32)
    p = Path(out_path)
    if p.exists():
        prev = pd.read_csv(p).set_index("file")
        for i, f in enumerate(files):
            if f in prev.index:
                row = prev.loc[f]
                t = int(row["times_in_test"])
                r = row["misclassification_rate"]
                tested[i] = t
                wrong[i]  = round(r * t)
    return wrong, tested


def save_misclas(out_path, wrong, tested, files, label_encoder, true_labels_np):
    misclas_rate = np.divide(wrong, tested, out=np.zeros_like(wrong, dtype=float),
                             where=tested > 0)
    pd.DataFrame({
        "file":                 files,
        "true_module":          label_encoder.inverse_transform(true_labels_np),
        "times_in_test":        tested,
        "misclassification_rate": misclas_rate.round(3),
    }).sort_values("misclassification_rate", ascending=False).to_csv(out_path, index=False)


def already_done(stem, model_type, ablation, results_file):
    p = Path(results_file)
    if not p.exists():
        return set()
    df = pd.read_csv(p)
    mask = (
        (df["data_name"] == stem) & (df["model"] == model_type.lower()) &
        (df["mode"] == ablation["mode"]) &
        (df["directed"] == ablation["directed"]) &
        (df["reverse_relations"] == ablation["reverse_relations"])
    )
    return set(df[mask]["run_id"].tolist())


print("Utilities ready.")

Utilities ready.


In [6]:
# ── Cell 4: RUN EXPERIMENTS ───────────────────────────────────────────────────

_discovered = set(datasets.discover_datasets(DATA_DIR))
stems = (
    [s for s in datasets.DISPLAY_NAMES if s not in ["ant",] and s in _discovered and s in STEM_TO_DISPLAY]
    if DATASETS == "all" else DATASETS
)
print(f"Datasets : {stems}")
print(f"Models   : {MODELS}")
print(f"Ablations: {[a['name'] for a in ABLATIONS]}")
print(f"Runs     : {len(RUN_IDS)}  ({RUN_IDS[0]}..{RUN_IDS[-1]})")
print(f"Flush every {FLUSH_EVERY} runs\n")


def run_pending(stem, model_type, mode, directed, rev_rel, base_data, in_channels,
                 num_classes, file_df, files, n_nodes, label_encoder, true_labels_np,
                 splits, results_file, misclas_path, mp, num_layers=None):
    ablation = {"mode": mode, "directed": directed, "reverse_relations": rev_rel}
    done    = already_done(stem, model_type, ablation, results_file)
    pending = [r for r in RUN_IDS if r not in done]
    if not pending:
        print(f"  [{model_type}] all {len(RUN_IDS)} runs already saved — skipping")
        return
    print(f"  [{model_type}] {len(done)} done, {len(pending)} to run")

    layers = num_layers or mp["num_layers"]

    track_misclas = misclas_path is not None
    if track_misclas:
        wrong, tested = load_misclas(misclas_path, n_nodes, files)

    buffer = []
    for run_id in pending:
        train_files = splits[str(run_id)]["train"]
        seed_idx    = split_to_index(train_files, file_df).to(DEVICE)
        if len(seed_idx) == 0:
            continue

        data_dev = copy.deepcopy(base_data).to(DEVICE)

        if model_type == "GCN":
            model = RelationalGCN(
                data_dev.metadata(), in_channels, mp["hidden"], num_classes,
                num_layers=layers, dropout=mp["dropout"], mode=mode,
            )
        elif model_type == "GAT":
            model = RelationalGAT(
                data_dev.metadata(), in_channels, mp["hidden"], num_classes,
                num_layers=layers, heads=mp["heads"],
                dropout=mp["dropout"], mode=mode,
            )
        else:  # MLP
            model = MLP(in_channels=in_channels, hidden_channels=mp["hidden"],
                        out_channels=num_classes, dropout=mp["dropout"])

        result = self_train(
            data_dev, model, seed_idx, lr=LR, device=DEVICE,
            threshold=THRESHOLD, warmup_epochs=WARMUP_EPOCHS,
            self_train_rounds=SELF_TRAIN_ROUNDS, self_train_epochs=SELF_TRAIN_EPOCHS,
            return_predictions=track_misclas,
        )
        result.pop("metrics_history", None)
        predictions = result.pop("predictions", [])

        if track_misclas:
            seed_set = set(seed_idx.cpu().tolist())
            for p in predictions:
                i = p["node_idx"]
                if i in seed_set:
                    continue
                tested[i] += 1
                if p["pred_label"] is None or p["pred_label"] != p["true_label"]:
                    wrong[i] += 1

        result.update({
            "dataset": datasets.display_name(stem), "data_name": stem,
            "model": model_type.lower(), "mode": mode,
            "directed": directed, "reverse_relations": rev_rel,
            "run_id": run_id,
            "threshold": THRESHOLD, "warmup_epochs": WARMUP_EPOCHS,
            "self_train_rounds": SELF_TRAIN_ROUNDS, "self_train_epochs": SELF_TRAIN_EPOCHS,
        })
        buffer.append(result)
        print(f"    run {run_id:3d}  f1_macro={result.get('f1_macro', 0):.3f}  "
              f"mapped_f1_macro={result.get('mapped_f1_macro', 0):.3f}  "
              f"coverage={result['coverage']:.3f}  "
              f"mapped={result['n_mapped_nodes']}/{result['n_test_nodes']}")

        if len(buffer) >= FLUSH_EVERY:
            flush(buffer, results_file)
            if track_misclas:
                save_misclas(misclas_path, wrong, tested, files, label_encoder, true_labels_np)
            buffer = []

    flush(buffer, results_file)
    if track_misclas:
        save_misclas(misclas_path, wrong, tested, files, label_encoder, true_labels_np)


# ---- Precompute per-dataset data once (features + splits), reused by every phase ----
dataset_cache = {}
for stem in stems:
    print(f"Loading {datasets.display_name(stem)}...")

    file_df, file_dep = datasets.load_file_level(stem, DATA_DIR)
    w2v = features_w2v.build_w2v_features(file_df, stem=stem, data_dir=DATA_DIR)
    loc = loc_features.build_loc_features(file_df)
    x   = torch.cat([loc, w2v], dim=1)
    in_channels = x.shape[1]

    label_encoder = LabelEncoder().fit(file_df["Module"])
    num_classes   = len(label_encoder.classes_)
    files         = file_df["File"].tolist()
    n_nodes       = len(files)
    true_labels_np = label_encoder.transform(file_df["Module"])

    print(f"  {n_nodes} files | {num_classes} modules | feat_dim={in_channels}")

    try:
        splits = load_splits(stem)
    except FileNotFoundError as e:
        print(f"  SKIP: {e}")
        continue

    dataset_cache[stem] = dict(
        file_df=file_df, file_dep=file_dep, x=x, in_channels=in_channels,
        label_encoder=label_encoder, num_classes=num_classes, files=files,
        n_nodes=n_nodes, true_labels_np=true_labels_np, splits=splits,
    )

active_stems = [s for s in stems if s in dataset_cache]

# ---- Phase 1: MLP for every dataset ---------------------------------------------
print(f"\n{'#'*60}")
print("### PHASE 1: MLP (all datasets)")
print(f"{'#'*60}")
for stem in active_stems:
    d = dataset_cache[stem]
    print(f"\n{'='*60}")
    print(f"Dataset: {datasets.display_name(stem)}")

    if "MLP" not in MODELS:
        continue

    mlp_data = HeteroData()
    mlp_data["file"].x = d["x"]
    mlp_data["file"].y = torch.tensor(d["true_labels_np"], dtype=torch.long)

    run_pending(
        stem, "MLP", mode="hetero", directed=True, rev_rel=False,
        base_data=mlp_data, in_channels=d["in_channels"], num_classes=d["num_classes"],
        file_df=d["file_df"], files=d["files"], n_nodes=d["n_nodes"],
        label_encoder=d["label_encoder"], true_labels_np=d["true_labels_np"],
        splits=d["splits"],
        results_file=results_file_for("MLP"),
        misclas_path=f"{NODE_PRED_DIR}/mlp_{datasets.display_name(stem)}_misclas.csv",
        mp=MODEL_PARAMS.get("MLP", {}),
    )

# ---- Phase 2: hetero_directed GCN/GAT for every dataset ------------------------
_hetero_ablation = next(a for a in ABLATIONS if a["name"] == "hetero_directed")
print(f"\n{'#'*60}")
print("### PHASE 2: hetero_directed GCN/GAT (all datasets)")
print(f"{'#'*60}")
for stem in active_stems:
    d = dataset_cache[stem]
    print(f"\n{'='*60}")
    print(f"Dataset: {datasets.display_name(stem)}")

    directed = _hetero_ablation["directed"]
    rev_rel  = _hetero_ablation["reverse_relations"]
    mode     = _hetero_ablation["mode"]

    gnn_data, _, _ = graph.build_graph(
        d["file_df"], d["file_dep"], d["x"], directed=directed, reverse_relations=rev_rel
    )
    gnn_data["file"].y = torch.tensor(d["true_labels_np"], dtype=torch.long)

    for model_type in [m for m in MODELS if m in ("GCN", "GAT")]:
        misclas_path = f"{NODE_PRED_DIR}/{model_type.lower()}_hetero_{datasets.display_name(stem)}_misclas.csv"
        run_pending(
            stem, model_type, mode=mode, directed=directed, rev_rel=rev_rel,
            base_data=gnn_data, in_channels=d["in_channels"], num_classes=d["num_classes"],
            file_df=d["file_df"], files=d["files"], n_nodes=d["n_nodes"],
            label_encoder=d["label_encoder"], true_labels_np=d["true_labels_np"],
            splits=d["splits"],
            results_file=results_file_for(model_type, "hetero_directed"),
            misclas_path=misclas_path,
            mp=MODEL_PARAMS.get(model_type, {}),
        )

# ---- Phase 3: remaining ablations (homo_directed, hetero_undirected, hetero_reversed, hetero_directed_1layer) ----
_remaining_ablations = [a for a in ABLATIONS if a["name"] != "hetero_directed"]
print(f"\n{'#'*60}")
print("### PHASE 3: remaining ablations GCN/GAT (all datasets)")
print(f"{'#'*60}")
for ablation in _remaining_ablations:
    directed = ablation["directed"]
    rev_rel  = ablation["reverse_relations"]
    mode     = ablation["mode"]
    layers_override = ablation.get("num_layers")

    print(f"\n{'-'*60}")
    print(f"-- Ablation: {ablation['name']}")

    for stem in active_stems:
        d = dataset_cache[stem]
        print(f"\n{'='*60}")
        print(f"Dataset: {datasets.display_name(stem)}")

        gnn_data, _, _ = graph.build_graph(
            d["file_df"], d["file_dep"], d["x"], directed=directed, reverse_relations=rev_rel
        )
        gnn_data["file"].y = torch.tensor(d["true_labels_np"], dtype=torch.long)

        for model_type in [m for m in MODELS if m in ("GCN", "GAT")]:
            run_pending(
                stem, model_type, mode=mode, directed=directed, rev_rel=rev_rel,
                base_data=gnn_data, in_channels=d["in_channels"], num_classes=d["num_classes"],
                file_df=d["file_df"], files=d["files"], n_nodes=d["n_nodes"],
                label_encoder=d["label_encoder"], true_labels_np=d["true_labels_np"],
                splits=d["splits"],
                results_file=results_file_for(model_type, ablation["name"]),
                misclas_path=None,
                mp=MODEL_PARAMS.get(model_type, {}),
                num_layers=layers_override,
            )

print("\nAll done.")

Datasets : ['argouml', 'commons', 'jabref', 'lucene', 'sweetHome', 'teammates', 'bash', 'hdc', 'hdf', 'chromium']
Models   : ['GCN', 'GAT', 'MLP']
Ablations: ['hetero_directed', 'homo_directed', 'hetero_undirected', 'hetero_directed_1layer']
Runs     : 100  (0..99)
Flush every 2 runs

Loading A.UML...
  766 files | 14 modules | feat_dim=167
Loading C.Img...
  329 files | 21 modules | feat_dim=181
Loading Jabref...
  1180 files | 6 modules | feat_dim=217
Loading Lucene...
  1054 files | 7 modules | feat_dim=246
Loading SH-3D...
  167 files | 9 modules | feat_dim=141
Loading T.Mates...
  778 files | 15 modules | feat_dim=159
Loading Bash...
  292 files | 13 modules | feat_dim=368
Loading HDC...
  207 files | 11 modules | feat_dim=310
Loading HDF...
  153 files | 9 modules | feat_dim=266
Loading Chromium...
  18343 files | 69 modules | feat_dim=11044

############################################################
### PHASE 1: MLP (all datasets)
##############################################

In [7]:
# ── Cell 4b: RUN NBA BASELINE — same fixed splits as GCN/GAT/MLP ──────────────
import time
from types import SimpleNamespace
from baselines.nba import prepare_dataset, run_iterative_nb_mapping, evaluate_mapping_results

NBA_CFG = json.loads((Path.cwd() / "experiments" / "config.json").read_text())["nba"]
NBA_RESULTS_FILE = f"{RESULTS_DIR}/nba_results.csv"

nba_config = SimpleNamespace(
    use_cda                 = NBA_CFG["use_cda"],
    use_node_text           = NBA_CFG["use_node_text"],
    use_node_name           = NBA_CFG["use_node_name"],
    use_arch_component_name = NBA_CFG["use_arch_component_name"],
    min_word_length          = 3,
    mapping_threshold        = NBA_CFG["mapping_threshold"],
    word_count               = NBA_CFG["word_count"],
    use_stemming              = True,
    random_seed               = None,
)


def already_done_nba(stem, results_file):
    p = Path(results_file)
    if not p.exists():
        return set()
    df = pd.read_csv(p)
    return set(df[df["dataset"] == datasets.display_name(stem)]["run_id"].tolist())


def flush_nba(rows, out_path):
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    p = Path(out_path)
    if p.exists():
        df_new = pd.concat([pd.read_csv(p), df_new], ignore_index=True).drop_duplicates(
            subset=["dataset", "run_id"], keep="first",
        )
    df_new.to_csv(p, index=False)


print(f"\n{'#'*60}")
print("### NBA BASELINE (all datasets, same splits as GCN/GAT/MLP)")
print(f"{'#'*60}")
for stem in active_stems:
    d = dataset_cache[stem]
    print(f"\n{'='*60}")
    print(f"Dataset: {datasets.display_name(stem)}")

    done    = already_done_nba(stem, NBA_RESULTS_FILE)
    pending = [r for r in RUN_IDS if r not in done]
    if not pending:
        print(f"  [NBA] all {len(RUN_IDS)} runs already saved — skipping")
        continue
    print(f"  [NBA] {len(done)} done, {len(pending)} to run")

    prepared = prepare_dataset(d["file_df"], d["file_dep"])
    splits   = d["splits"]

    buffer = []
    for run_id in pending:
        train_files = splits[str(run_id)]["train"]
        t0 = time.time()
        results, file_labels, stats = run_iterative_nb_mapping(prepared, train_files, nba_config)
        elapsed = time.time() - t0
        metrics = evaluate_mapping_results(results, file_labels)
        if metrics is None:
            continue
        row = {
            **metrics,
            "dataset": datasets.display_name(stem), "data_name": stem, "run_id": run_id,
            "train_size": len(train_files), "elapsed_seconds": round(elapsed, 3),
            **stats, "mapping_threshold": nba_config.mapping_threshold,
        }
        buffer.append(row)
        print(f"    run {run_id:3d}  f1_macro={row.get('f1_macro', 0):.3f}  "
              f"coverage={row['coverage']:.3f}")

        if len(buffer) >= FLUSH_EVERY:
            flush_nba(buffer, NBA_RESULTS_FILE)
            buffer = []

    flush_nba(buffer, NBA_RESULTS_FILE)

print("\nNBA done.")


############################################################
### NBA BASELINE (all datasets, same splits as GCN/GAT/MLP)
############################################################

Dataset: A.UML
  [NBA] all 100 runs already saved — skipping

Dataset: C.Img
  [NBA] all 100 runs already saved — skipping

Dataset: Jabref
  [NBA] all 100 runs already saved — skipping

Dataset: Lucene
  [NBA] all 100 runs already saved — skipping

Dataset: SH-3D
  [NBA] all 100 runs already saved — skipping

Dataset: T.Mates
  [NBA] all 100 runs already saved — skipping

Dataset: Bash
  [NBA] 23 done, 77 to run

Dataset: HDC
  [NBA] 99 done, 1 to run

Dataset: HDF
  [NBA] 97 done, 3 to run

Dataset: Chromium
  [NBA] 56 done, 44 to run
    run  56  f1_macro=0.021  coverage=0.949
    run  57  f1_macro=0.022  coverage=0.961
    run  58  f1_macro=0.022  coverage=0.964
    run  59  f1_macro=0.025  coverage=0.961
    run  60  f1_macro=0.020  coverage=0.955
    run  61  f1_macro=0.022  coverage=0.942
    run  

KeyboardInterrupt: 

In [ ]:
# ── Cell 5: SUMMARY ───────────────────────────────────────────────────────────
# Point RESULTS_FILE at whichever per-model/per-ablation CSV you want to inspect,
# e.g. results_file_for("GCN", "hetero_directed") or results_file_for("MLP").

RESULTS_FILE = results_file_for("GCN", "hetero_directed")
df = pd.read_csv(RESULTS_FILE)
print(f"Total rows: {len(df)}")

summary = (
    df.groupby(["dataset", "model", "mode"])[
        ["f1_macro", "f1_micro", "coverage"]
    ]
    .agg(["mean", "std"])
    .round(3)
)
summary